# Tutorial 2: Integrating all 12 DLPFC slices

This tutorial integrates 12 human dorsolateral prefrontal cortex (DLPFC) Visium slices with **SpaRMA**. It covers data loading, preprocessing, spatial-graph construction, the two training stages, clustering, and visualization.

## 1. Preparation

Install SpaRMA and a PyTorch/PyTorch Geometric build suitable for your system. The expected directory for each section is `Data/<section_id>/`, containing the standard Space Ranger output (`filtered_feature_bc_matrix.h5` and `spatial/`).

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch

from spaarma import fit, get_preset
from spaarma.graph import spatial_edges

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Load data

Each barcode is suffixed with its section identifier so that observation names remain unique after concatenation. A slice identifier is stored in `obs['batch_name']`.

In [ ]:
data_root = Path("Data")
section_ids = ['151507', '151508', '151509', '151510', '151669', '151670', '151671', '151672', '151673', '151674', '151675', '151676']

slices = []
for section_id in section_ids:
    sample = sc.read_visium(
        data_root / section_id,
        count_file=f"{section_id}_filtered_feature_bc_matrix.h5",
        load_images=True,
    )
    sample.var_names_make_unique()
    sample.obs_names = [f"{barcode}_{section_id}" for barcode in sample.obs_names]
    sample.obs["batch_name"] = section_id
    sample.obs["batch_name"] = sample.obs["batch_name"].astype("category")
    slices.append(sample)
    print(section_id, sample.n_obs, "spots", sample.n_vars, "genes")

## 3. Preprocess each slice

Highly variable genes are identified independently in each slice from count data. The shared set is retained, after which every slice is library-size normalized and log transformed. The same ordered gene set is used for all slices.

In [ ]:
hvg_sets = []
for sample in slices:
    sc.pp.highly_variable_genes(sample, flavor="seurat_v3", n_top_genes=10000)
    hvg_sets.append(set(sample.var_names[sample.var["highly_variable"]]))

shared_genes = sorted(set.intersection(*hvg_sets))
if not shared_genes:
    raise RuntimeError("No shared highly variable genes were found.")

processed = []
for sample in slices:
    sample = sample[:, shared_genes].copy()
    sc.pp.normalize_total(sample, target_sum=1e4)
    sc.pp.log1p(sample)
    processed.append(sample)

adata = ad.concat(processed, join="inner", merge="same", uns_merge="unique")
adata.obs["batch_name"] = adata.obs["batch_name"].astype("category")
print(adata)
print("Shared genes:", adata.n_vars)

## 4. Concatenate the Scanpy objects and spatial networks

Edges are generated independently within each slice from the physical coordinates. Self-loops are added by the graph builder, and no cross-slice edge is inserted at this step.

In [ ]:
preset = get_preset("dlpfc12")
batches = adata.obs[preset.batch_key].astype(str).to_numpy()
edge_index = spatial_edges(adata.obsm["spatial"], batches, preset.radius)
print("Directed edges including self-loops:", edge_index.shape[1])

## 5. Run SpaRMA

Stage I performs masked spot reconstruction. Stage II restores clean inputs and jointly optimizes expression reconstruction and multi-positive attention alignment. The returned representation has 30 dimensions.

In [ ]:
x = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
embedding, settings, model = fit(
    x=x,
    edge_index=edge_index,
    batches=batches,
    config=preset.training,
    device=device,
)
adata.obsm["SpaRMA"] = embedding
adata.uns["SpaRMA"] = settings
print("Embedding shape:", embedding.shape)

## 6. Clustering

Clustering and UMAP are performed only after training. The example uses `mclust` with the EEE covariance model and seven clusters. This cell requires R, `mclust`, and `rpy2`.

In [ ]:
def mclust_embedding(adata, n_clusters=7, model="EEE", random_seed=0):
    """Cluster the learned embedding after training; annotations are not used."""
    import rpy2.robjects as ro
    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
    ro.r("library(mclust)")
    ro.r["set.seed"](random_seed)
    result = ro.r["Mclust"](
        adata.obsm["SpaRMA"], n_clusters, modelNames=model
    )
    labels = np.asarray(result.rx2("classification"), dtype=int).astype(str)
    adata.obs["SpaRMA_domain"] = labels
    return labels

mclust_embedding(adata, n_clusters=7)
sc.pp.neighbors(adata, use_rep="SpaRMA")
sc.tl.umap(adata, random_state=0)

## 7. Visualization

In [ ]:
sc.pl.umap(adata, color=["batch_name", "SpaRMA_domain"], wspace=0.35)

fig, axes = plt.subplots(1, len(adata.obs["batch_name"].cat.categories), figsize=(4 * len(adata.obs["batch_name"].cat.categories), 4))
axes = np.atleast_1d(axes)
for ax, batch in zip(axes, adata.obs["batch_name"].cat.categories):
    view = adata[adata.obs["batch_name"] == batch]
    sc.pl.embedding(view, basis="spatial", color="SpaRMA_domain", title=batch, ax=ax, show=False, size=18)
plt.tight_layout()